# Run both row-sum notebooks

One cell. It executes `ROWSUM_CLCD_logit.ipynb` and
`ROWSUM_CLCD_attention_level.ipynb` in turn, in this same Python environment,
writes `rowsum_<rule>.csv` / `.json` beside them, and prints the two side by side.

Inference only. No training. Expect a few minutes per rule on the GPU.

If this cell cannot find `jupyter`, open each `ROWSUM_CLCD_*.ipynb` and press
Run All instead -- the result is identical.


In [2]:
import os, sys, subprocess, time, json

FIXED = r"D:\Master Thesis Codes\attention\ROWSUM"
HERE  = os.getcwd()
CANDS = [FIXED, HERE, os.path.join(HERE, "ROWSUM")]
ROOT  = next((c for c in CANDS if os.path.exists(os.path.join(c, "ROWSUM_CLCD_logit.ipynb"))), FIXED)
ORDER = ["logit", "attention_level"]

print("working in:", ROOT)
ok, bad = [], []
for v in ORDER:
    nb = os.path.join(ROOT, "ROWSUM_CLCD_%s.ipynb" % v)
    if not os.path.exists(nb):
        bad.append((v, "notebook not found")); print("SKIP %-16s not found" % v); continue
    print("\n" + "=" * 70); print("RUN  %s" % v); print("=" * 70, flush=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
                        "--inplace", "--ExecutePreprocessor.timeout=3600", nb], capture_output=True, text=True)
    if r.returncode == 0: ok.append(v); print("done in %.0f s" % (time.time() - t0))
    else: bad.append((v, (r.stderr or "")[-1500:])); print("FAILED after %.0f s" % (time.time() - t0))

print("\n" + "=" * 70)
print("succeeded: %s" % (", ".join(ok) or "none"))
for v, why in bad: print("\nFAILED %s\n%s" % (v, why))

R = {}
for v in ORDER:
    p = os.path.join(ROOT, "rowsum_%s.json" % v)
    if os.path.exists(p): R[v] = json.load(open(p, encoding="utf-8"))
if R:
    rows = [("mean |row sum - 1|", "mean_abs_dev", "%.6f"), ("max |row sum - 1|", "max_abs_dev", "%.4f"),
            ("smallest row sum", "min", "%.4f"), ("largest row sum", "max", "%.4f"),
            ("rows within +-0.01 of 1 (%)", "pct_within_0.01", "%.2f"), ("rows within +-0.05 of 1 (%)", "pct_within_0.05", "%.2f"),
            ("rows within +-0.10 of 1 (%)", "pct_within_0.10", "%.2f"),
            ("output change if rows renormalized, before LayerNorm", "rel_change_before_layernorm", "%.5f"),
            ("output change if rows renormalized, after LayerNorm", "rel_change_after_layernorm", "%.5f")]
    print("\n" + "=" * 100)
    print(" Row sums of the attention maps, all blocks, heads and both directions -- CLCD test split")
    print("=" * 100)
    print("%-56s %20s %20s" % ("", "logit (ours)", "attention_level"))
    print("-" * 100)
    for label, key, fmt in rows:
        print("%-56s %20s %20s" % (label, (fmt % R["logit"]["maps"]["all"][key]) if "logit" in R else "-",
                                   (fmt % R["attention_level"]["maps"]["all"][key]) if "attention_level" in R else "-"))
    print("=" * 100)
    for v in R: print(" %s: %d pairs, %d rows, parity %.1e" % (v, R[v]["pairs"], R[v]["maps"]["all"]["rows"], R[v]["parity_max_abs_delta"]))


working in: D:\Master Thesis Codes\attention\ROWSUM

RUN  logit
done in 161 s

RUN  attention_level
done in 143 s

succeeded: logit, attention_level

 Row sums of the attention maps, all blocks, heads and both directions -- CLCD test split
                                                                 logit (ours)      attention_level
----------------------------------------------------------------------------------------------------
mean |row sum - 1|                                                   0.000000             0.208062
max |row sum - 1|                                                      0.0000              69.3832
smallest row sum                                                       1.0000               0.5018
largest row sum                                                        1.0000              70.3832
rows within +-0.01 of 1 (%)                                            100.00                 4.06
rows within +-0.05 of 1 (%)                                      